# Intrinsic Evaluation

Runs one selected intrinsic study from `src/evaluate/intrinsic.yaml` with diagnostics enabled.

## Environment

In [ ]:
import json
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import format_log_path


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


LOG_DIR = PROJECT_ROOT / "data" / "evaluation" / ".logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"intrinsic-gen-{datetime.now():%Y-%m-%d-%H-%M-%S}.log"


def log(message: str) -> None:
    line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as file:
        file.write(f"{line}\n")


def resolve_run_indices(n_runs: int, run_start: int = 1, run_count: int | None = None) -> list[int]:
    if n_runs <= 0 or run_start <= 0:
        raise ValueError("run settings must be positive")
    count = n_runs - run_start + 1 if run_count is None else int(run_count)
    if count <= 0 or run_start + count - 1 > n_runs:
        raise ValueError("requested run range is outside n_runs")
    return list(range(run_start, run_start + count))

## Configuration

In [ ]:
import copy
import shutil
import yaml

from src.evaluate import (
    build_method_dir,
    format_log_path,
    load_experiment_configuration,
    load_judge_configuration,
)

eval_type = "intrinsic"
EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "intrinsic.yaml"
eval_config = load_experiment_configuration(eval_type)
judge_config = load_judge_configuration()
evaluation_dir = PROJECT_ROOT / "data" / "evaluation"

study = str(eval_config.get("study", "encoder")).strip().lower()
if study not in {"encoder", "intervention", "safety"}:
    raise ValueError("intrinsic study must be encoder, intervention, or safety")
study_config = eval_config["studies"][study]
model_key = str(eval_config["model_key"])
editing_methods = list(eval_config.get("editing_methods", ["mnemo"]))
if editing_methods != ["mnemo"]:
    raise ValueError("Intrinsic evaluation supports Mnemo only")
result_names = list(eval_config.get("result_names", ["es_loc", "port"]))
eval_milestones = list(study_config["eval_milestones"])
n_runs = int(study_config.get("n_runs", eval_config.get("n_runs", 3)))
run_start = int(study_config.get("run_start", eval_config.get("run_start", 1)))
raw_run_count = study_config.get("run_count", eval_config.get("run_count"))
run_count = None if raw_run_count is None else int(raw_run_count)
run_indices = resolve_run_indices(n_runs, run_start, run_count)
judge_repeats = int(judge_config["judge"]["repeats"])
base_seed = int(eval_config["base_seed"])
generation_config = copy.deepcopy(eval_config["generation"])
# Intrinsic conditions are evaluated independently; intervention fan-out is synchronous.
if study == "intervention":
    generation_config.setdefault("async_generation", {})["enabled"] = False

shared_overrides = copy.deepcopy(eval_config.get("shared", {}).get("overrides", {}))
study_shared = copy.deepcopy(study_config.get("shared", {}).get("overrides", {}))

def deep_merge(base, override):
    result = copy.deepcopy(base)
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = copy.deepcopy(value)
    return result

def condition_for_run(run_index: int):
    runs = study_config.get("runs", [])
    if not 1 <= run_index <= len(runs):
        raise ValueError(f"No intrinsic condition for run {run_index}")
    return runs[run_index - 1]

def method_slot(run_index: int, *, strength: float | None = None) -> str:
    if study != "intervention":
        return f"mnemo_{study}"
    if strength is None:
        raise ValueError("intervention method slot requires strength")
    return f"mnemo_intervention_s{int(round(strength * 100)):03d}"

def condition_config_path(run_index: int) -> Path:
    condition = condition_for_run(run_index)
    overrides = deep_merge(shared_overrides, study_shared)
    overrides = deep_merge(overrides, condition.get("overrides", {}))
    base_path = PROJECT_ROOT / "src" / "config" / "mnemo" / f"{model_key}.yaml"
    base = yaml.safe_load(base_path.read_text(encoding="utf-8"))
    merged = deep_merge(base, overrides)
    # The temporary file lives outside the editor config tree; preserve the
    # model-config reference as an absolute path so relative resolution remains valid.
    model_config_ref = merged.get("model_config")
    if isinstance(model_config_ref, str):
        merged["model_config"] = str((base_path.parent / model_config_ref).resolve())
    config_dir = PROJECT_ROOT / "temp" / "intrinsic_configs"
    config_dir.mkdir(parents=True, exist_ok=True)
    path = config_dir / f"{study}-r{run_index}.yaml"
    path.write_text(yaml.safe_dump(merged, sort_keys=False), encoding="utf-8")
    return path

def write_condition_metadata(run_index: int, slot: str, *, strength: float | None, result_name: str, cohort_metadata: dict):
    condition = condition_for_run(run_index)
    method_dir = build_method_dir(model_key, slot, eval_milestones, n_runs, judge_repeats, evaluation_dir, eval_type)
    metadata_dir = method_dir / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    path = metadata_dir / f"condition.{run_index}.json"
    existing = {}
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
    cohorts = dict(existing.get("cohorts", {}))
    cohorts[result_name] = cohort_metadata
    payload = {
        "eval_type": eval_type, "study": study, "condition_id": condition.get("id"),
        "run_index": run_index, "model_key": model_key, "method": slot,
        "method_slot": slot, "strength": strength,
        "overrides": condition.get("overrides", {}),
        "shared_overrides": deep_merge(shared_overrides, study_shared),
        "cohorts": cohorts,
        "timing_shared": study == "intervention",
    }
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")

set_seed(base_seed)
log(f"[config] loaded={format_log_path(EVAL_CONFIG_PATH)} study={study} model={model_key} runs={run_indices} milestones={eval_milestones}")

## Datasets

In [ ]:
from collections.abc import Mapping
from datasets import load_dataset
from src.editors import EditRequest
from src.evaluate import EvalRecord, EvalResult, MetricResult, select_fixed_cohort

es_loc_data = load_dataset(eval_config["dataset_repo"], "es_loc", split="data")
port_data = load_dataset(eval_config["dataset_repo"], "port", split="data")
COHORT_SIZE = eval_milestones[-1]
COHORTS, COHORT_METADATA = {}, {}
for name, dataset in (("es_loc", es_loc_data), ("port", port_data)):
    COHORTS[name], COHORT_METADATA[name] = select_fixed_cohort(dataset, name, COHORT_SIZE, base_seed, {"es_loc": 17, "port": 31}[name])

def build_es_loc_data(record: Mapping):
    edit, answer = record["edit"], record["edit"]["answer"]
    request = EditRequest(edit["subject"], edit["prompt"], answer["new"]["str"])
    es = [EvalRecord(edit["prompt"], [answer["new"]["str"]])] + [EvalRecord(p, [answer["new"]["str"]]) for p in record["generality"]]
    loc = [EvalRecord(item["prompt"], [item["answer"]["str"]]) for item in record["locality"]]
    return request, EvalResult(success=MetricResult(details=es), locality=MetricResult(details=loc))

def build_port_data(record: Mapping):
    edit, answer = record["edit"], record["edit"]["answer"]
    request = EditRequest(edit["subject"]["str"], edit["prompt"], answer["new"]["str"])
    port = [EvalRecord(prompt, record["portability"]["answers"]["new"]) for prompt in record["portability"]["prompts"]]
    return request, EvalResult(portability=MetricResult(details=port))

def aggregate_eval_results(results):
    if not results: return EvalResult()
    def agg(name):
        vals = [getattr(r, name) for r in results]; vals = [v for v in vals if v is not None]
        return None if not vals else MetricResult(score=float(np.mean([v.score for v in vals])), details=[d for v in vals for d in v.details])
    return EvalResult(success=agg("success"), locality=agg("locality"), portability=agg("portability"))

## Benchmark Engine

In [ ]:
from time import perf_counter
from tqdm.notebook import trange
from src.editors import load_editor
from src.evaluate import (
    generate_and_save_unscored,
    save_memory_results, save_time_results, save_run_hardware,
)

GLOBAL_MODEL = None
GLOBAL_TOKENIZER = None

def get_editor_config_path(run_index: int) -> Path:
    return condition_config_path(run_index)

def set_runtime_strength(editor, strength: float) -> None:
    setter = getattr(editor, "set_intervention_strength", None)
    if callable(setter):
        setter(strength)
        return
    runtime = getattr(editor, "model", None)
    if runtime is None or not hasattr(runtime, "intervention_strength"):
        raise RuntimeError("Mnemo runtime does not expose intervention strength")
    runtime.intervention_strength = float(strength)

def run_condition(editor, edit_requests, eval_results, result_name: str, run_index: int, config_path: Path):
    max_step = eval_milestones[-1]
    milestones = set(eval_milestones)
    total = 0.0
    edit_times = []
    strengths = [float(s) for s in study_config.get("strengths", [])] if study == "intervention" else [None]
    slots = [method_slot(run_index, strength=s) for s in strengths] if study == "intervention" else [method_slot(run_index)]
    for step in trange(1, max_step + 1, desc=f"Intrinsic [{study}/{result_name}] r{run_index}", leave=False):
        start = perf_counter()
        editor.edit(edit_requests[step - 1])
        elapsed = perf_counter() - start
        edit_times.append(elapsed)
        total += elapsed
        for event in editor.drain_log_events():
            log(f"[editor-event] run={run_index}/{n_runs} method=mnemo result={result_name} step={step}/{max_step} payload={json.dumps(event, sort_keys=True)}")
        if step not in milestones:
            continue
        if study == "intervention":
            for strength, slot in zip(strengths, slots):
                set_runtime_strength(editor, strength)
                generate_and_save_unscored(
                    model=editor.model, tokenizer=editor.tokenizer,
                    eval_results=eval_results, edit_requests=edit_requests,
                    editing_method=slot, result_name=result_name, step=step,
                    max_step=max_step, run_index=run_index, model_key=model_key,
                    milestones=eval_milestones, n_runs=n_runs, judge_repeats=judge_repeats,
                    generation_config=generation_config, aggregate_eval_results=aggregate_eval_results,
                    log_fn=log, evaluation_dir=evaluation_dir, eval_type=eval_type,
                )
        else:
            generate_and_save_unscored(
                model=editor.model, tokenizer=editor.tokenizer,
                eval_results=eval_results, edit_requests=edit_requests,
                editing_method=method_slot(run_index), result_name=result_name, step=step,
                max_step=max_step, run_index=run_index, model_key=model_key,
                milestones=eval_milestones, n_runs=n_runs, judge_repeats=judge_repeats,
                generation_config=generation_config, aggregate_eval_results=aggregate_eval_results,
                log_fn=log, evaluation_dir=evaluation_dir, eval_type=eval_type,
            )
    diagnostics = {
        "external_memory_size": getattr(editor, "external_memory_size", None),
        "consolidation_history": list(getattr(editor, "consolidation_history", [])),
        "final_external_only": getattr(editor, "external_only", None),
    }
    return edit_times, total / max_step, diagnostics

def run_intrinsic(result_name, build_data):
    global GLOBAL_MODEL, GLOBAL_TOKENIZER
    for run_index in run_indices:
        slots = [method_slot(run_index, strength=s) for s in study_config.get("strengths", [])] if study == "intervention" else [method_slot(run_index)]
        ordered = list(COHORTS[result_name])
        order_metadata = {
            **COHORT_METADATA[result_name],
            "fixed_across_conditions": True,
        }
        built = [build_data(record) for record in ordered]
        requests = [x[0] for x in built]
        results = [x[1] for x in built]
        config_path = get_editor_config_path(run_index)
        editor = load_editor(str(config_path), model=GLOBAL_MODEL, tokenizer=GLOBAL_TOKENIZER)
        try:
            if GLOBAL_MODEL is None or GLOBAL_TOKENIZER is None:
                GLOBAL_MODEL = getattr(editor, "_base_model", editor.model)
                GLOBAL_TOKENIZER = editor.tokenizer
            edit_times, avg_time, diagnostics = run_condition(editor, requests, results, result_name, run_index, config_path)
            # Timing is shared only across intervention strengths for this result cohort.
            source_slot = slots[0]
            source = save_time_results(
                edit_times, model_key, source_slot, eval_milestones,
                result_name, run_index, n_runs, judge_repeats,
                evaluation_dir, eval_type,
            )
            if study == "intervention":
                for slot in slots[1:]:
                    target = save_time_results(
                        edit_times, model_key, slot, eval_milestones,
                        result_name, run_index, n_runs, judge_repeats,
                        evaluation_dir, eval_type,
                    )
                    shutil.copyfile(source, target)
            # Preserve machine-readable diagnostics and hardware for every artifact slot.
            for slot in slots:
                write_condition_metadata(
                    run_index,
                    slot,
                    strength=(None if study != "intervention" else float(slot.split("s")[-1]) / 100),
                    result_name=result_name,
                    cohort_metadata={"order": order_metadata},
                )
                save_memory_results(
                    memory={"edit": diagnostics}, model_key=model_key,
                    editing_method=slot, milestones=eval_milestones,
                    result_name=result_name, run_index=run_index,
                    n_runs=n_runs, judge_repeats=judge_repeats,
                    evaluation_dir=evaluation_dir, eval_type=eval_type,
                )
                save_run_hardware(
                    model_key=model_key, editing_method=slot,
                    milestones=eval_milestones, result_name=result_name,
                    run_index=run_index, n_runs=n_runs,
                    judge_repeats=judge_repeats,
                    evaluation_dir=evaluation_dir, eval_type=eval_type,
                )
        finally:
            editor.reset()
            del editor

## Edit Success And Locality

In [ ]:
if "es_loc" in result_names:
    run_intrinsic("es_loc", build_es_loc_data)
else:
    log("[skip] result=es_loc is not enabled in result_names")

## Portability

In [ ]:
if "port" in result_names:
    run_intrinsic("port", build_port_data)
else:
    log("[skip] result=port is not enabled in result_names")